Never used the github API before so just having a little explore to work things out#

In [2]:
import requests
import logging
import uuid
import json
from datetime import datetime
from google.cloud import storage

In [3]:
r=requests.get('https://stats.tennismylife.org/api/data-files')
latest_2026_file_from_source=next(i for i in r.json()['files'] if "2026" in i['name'])
last_modified=latest_2026_file_from_source['mtime']
csv_url=latest_2026_file_from_source['url']


Writing a file into the bucket

In [3]:
client=storage.Client()
bucket=client.bucket('tennisdataengproject-landing')

In [ ]:
ingest_datetime=datetime.now()
ingest_date=ingest_datetime.strftime("%Y-%m-%d")
batch_id=str(uuid.uuid4())

In [ ]:
data_path=f"raw/tml/matches/ingest_date={ingest_date}/batch_id={batch_id}/2026.csv"
manifest_path=f"raw/tml/matches/ingest_date={ingest_date}/batch_id={batch_id}/_manifest.json"

In [ ]:
manifest_dict={
    "batch_id":batch_id,
    "ingested_at":ingest_datetime
}

In [13]:
logger=logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)a %(levelname)s %(message)s"
)

In [ ]:
blobs=bucket.list_blobs(prefix='raw/tml/matches')
manifest_blobs=[i for i in blobs if i.name.endswith("_manifest.json")]
if not manifest_blobs:
    logger.info("No previous manifest found - First run")
    last_seen_mtime=None
else:
    latest=max(manifest_blobs, key=lambda x: x.time_created)
    last_seen_mtime=json.loads(latest.download_as_text())['source_version']

In [14]:
r=requests.get('https://stats.tennismylife.org/api/data-files', timeout=10)
latest_2026_file_from_source=next(i for i in r.json()['files'] if "2026" in i['name'])
last_modified=latest_2026_file_from_source['mtime']

In [17]:
type(last_modified)

str

In [ ]:
datetime.parse_string(last_modified)

TypeError: 'str' object cannot be interpreted as an integer

In [19]:
blobs=bucket.list_blobs(prefix='raw/tml/matches')
manifest_blobs=[i for i in blobs if i.name.endswith("_manifest.json")]

In [21]:
latest=max(manifest_blobs, key=lambda x: x.time_created)

In [30]:
latest.__dict__

{'name': 'raw/tml/matches/ingest_date=2026-08-29/batch_id=852f130d-35b0-42d3-951a-d10ad7d956a8/_manifest.json',
 '_properties': {'kind': 'storage#object',
  'id': 'tennisdataengproject-landing/raw/tml/matches/ingest_date=2026-08-29/batch_id=852f130d-35b0-42d3-951a-d10ad7d956a8/_manifest.json/1787999353746705',
  'selfLink': 'https://www.googleapis.com/storage/v1/b/tennisdataengproject-landing/o/raw%2Ftml%2Fmatches%2Fingest_date=2026-08-29%2Fbatch_id=852f130d-35b0-42d3-951a-d10ad7d956a8%2F_manifest.json',
  'mediaLink': 'https://storage.googleapis.com/download/storage/v1/b/tennisdataengproject-landing/o/raw%2Ftml%2Fmatches%2Fingest_date=2026-08-29%2Fbatch_id=852f130d-35b0-42d3-951a-d10ad7d956a8%2F_manifest.json?generation=1787999353746705&alt=media',
  'name': 'raw/tml/matches/ingest_date=2026-08-29/batch_id=852f130d-35b0-42d3-951a-d10ad7d956a8/_manifest.json',
  'bucket': 'tennisdataengproject-landing',
  'generation': '1787999353746705',
  'metageneration': '1',
  'contentType': 'appl

In [20]:
from google.cloud import bigquery


bucket=client.bucket('tennisdataengproject-landing')
bq_client = bigquery.Client()



# Check we have loaded the correct number of rows
verify_sql="""

SELECT COUNT(*)
FROM `tennisdataengproject.tml_raw.matches`
WHERE _batch_id=@batch_id


"""

verify_job_config=bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("batch_id","STRING",batch_id),
    ]
)
verify_result=bq_client.query(verify_sql, job_config=verify_job_config).result()

In [23]:
list(verify_result)[0]

ValueError: ('Iterator has already started', <google.cloud.bigquery.table.RowIterator object at 0x000001FBA648C6B0>)